# Leitura de dados

In [0]:

df_logs = spark.read.csv(
    "/Volumes/workspace/default/dados-streaming/logs_streaming.csv",
    header=True,
    inferSchema=False
)
df_catalog = spark.read.csv(
    "/Volumes/workspace/default/dados-streaming/catalog_files.csv",
    header=True,
    inferSchema=False
)
df_usuarios = spark.read.csv(
    "/Volumes/workspace/default/dados-streaming/usuarios.csv",
    header=True,
    inferSchema=False
)

# Tratamento de inconsistências

In [0]:
from pyspark.sql.functions import col, lower, when, expr, to_date

# Cast e normalização de campos base
df_logs = df_logs.withColumn(
    "watch_time_minutes",
    expr(
        """
        try_cast(
            replace(watch_time_minutes, 'min', ''),
            'int'
        )
        """
    )
)

df_logs = df_logs.withColumn(
    "playback_status",
    lower(col("playback_status"))
)

# Derivação de colunas
df_logs = df_logs.withColumn(
    "watch_time_hours",
    col("watch_time_minutes") / 60
)

df_logs = df_logs.withColumn(
    "watch_categories",
    when(col("watch_time_minutes") >= 120, "longo").otherwise("curto")
)

# Correção de datas
df_logs = df_logs.withColumn(
    "watch_date",
    to_date("watch_date", "yyyy-MM-dd")
)


In [0]:
from pyspark.sql.functions import when, col

df_final = df_final.withColumn(
    "genre",
    when(col("genre") == "Terror", "Horror")
    .when(col("genre") == "Animação", "Animation")
    .when(col("genre") == "Ação", "Action")
    .when(col("genre") == "Documentário", "Documentary")
    .when(col("genre") == "Sci-Fi", "Sci-fi")
    .when(col("genre") == "Comédia", "Comedy")
    .when(col("genre") == "Romance", "Romance")
    .when(col("genre") == "Drama", "Drama")
    .when(col("genre") == "Suspense", "Suspense")
    .otherwise(col("genre"))
)

# Agregações do DataFrame

In [0]:
from pyspark.sql.functions import col, count, avg, sum

df_logs.groupBy("subscription_type").agg(
    count("*").alias("total_sessoes"),
    avg("watch_time_minutes").alias("media_minutos_assistidos"),
    sum("watch_time_minutes").alias("total_minutos_assistidos")
).display()

df_logs.groupBy("subscription_type").agg(
    count("*").alias("total_sessoes"),
    avg("watch_time_minutes").alias("media_minutos_assistidos"),
    sum("watch_time_minutes").alias("total_minutos_assistidos")
).orderBy(col("total_sessoes").desc()).display()

df_logs.groupBy("country").agg(
    count("*").alias("total_sessoes"),
    avg("watch_time_minutes").alias("media_minutos_assistidos")
).orderBy(col("total_sessoes").desc()).display()


# Enriquecimento do DataFrame

In [0]:
df_enriquecido = df_logs.join(
    df_catalog,
    on="movie_id",
    how="left"
)

df_final = df_enriquecido.join(
    df_usuarios,
    on="user_id",
    how="left"
)

display(df_final)
